# Harmonic Analysis, Written as in Your Book
### Fourier series and the Fourier transform, every step checked by Lean

This notebook follows the opening chapters of the standard texts — E. M. Stein and R. Shakarchi,
*Fourier Analysis: An Introduction* (Princeton Lectures in Analysis I), and T. Tao's lecture notes on
harmonic analysis — and lets you write their proofs **as they are written there**, one sentence per step,
with **Lean** and **Mathlib** checking each step. (The books are the reference for the topics and the
conventions; the proofs below are written for this notebook.)

* ✅ means Lean accepted every step, with only the standard axioms (`propext`, `Classical.choice`,
  `Quot.sound`).
* ❌ comes with the number of the step Lean could not justify.
* The big theorems (Parseval, Riemann–Lebesgue, Fourier inversion, the Gaussian) are cited by name; each
  is proved from Mathlib inside every Lean file that uses it — nothing is assumed.

**Contents:** 1. Conventions · 2. Vocabulary and theorems · 3. Pictures (partial sums and Gibbs,
decay, Riemann–Lebesgue, the Gaussian) · 4. Worked examples (smoothness ⇒ decay, Riemann–Lebesgue from
Bessel, uniqueness from inversion) · 5. A wrong step · 6. Exercises · 7. What is not covered yet ·
8. Solutions

In [ ]:
import pyleanx as pl
from pyleanx import harmonic as ha
from pyleanx import complex_analysis as ca
from harmonic_lecture import *      # statements, figures, check()

# Letters already defined: f, g, h : ℝ → ℂ;  n : ℤ;  x, xi (ξ) : ℝ
# R(n): n as a real number;  Cx(n): n as a complex number;  ca.I(): the imaginary unit i

---
## 1. Conventions

| Object | Convention (as in Stein–Shakarchi) | PyLeanX | Lean / Mathlib |
|---|---|---|---|
| Fourier coefficient | $\hat f(n) = \frac{1}{2\pi}\int_{-\pi}^{\pi} f(x)\,e^{-inx}\,dx$ | `ha.coeff(f, n)` | `fourierCoeffOn` on $[-\pi,\pi]$ |
| Fourier transform | $\hat f(\xi) = \int_{\mathbb R} f(x)\,e^{-2\pi i x\xi}\,dx$ | `ha.ft(f, xi)`, `ha.ft_fun(f)` | `𝓕` |
| Inverse transform | $F^\vee(x) = \int_{\mathbb R} F(\xi)\,e^{2\pi i x\xi}\,d\xi$ | `ha.ift(F, x)` | `𝓕⁻` |
| $f\in L^2(-\pi,\pi)$, $L^1(-\pi,\pi)$, $L^1(\mathbb R)$ | | `ha.L2(f)`, `ha.integrable_on_circle(f)`, `ha.integrable(f)` | `MemLp`, `IntervalIntegrable`, `Integrable` |
| $\sum_{n}|\hat f(n)|^2$ and $\frac1{2\pi}\int_{-\pi}^{\pi}|f|^2$ | the two sides of Parseval | `ha.coeff_sq_sum(f)`, `ha.mean_square(f)` | `tsum`, interval integral |
| $f' = g$ | | `ha.derivative(f, g)` | `∀ x, HasDerivAt f (g x) x` |

The first two rows are theorems you can cite (`ha.coeff_eq_integral`, `ha.ft_eq_integral`): the
definitions agree with the formulas in the book. Run the next cell to see them as Lean states them.

In [ ]:
print(ha.coeff_eq_integral.lean, "\n")
print(ha.ft_eq_integral.lean)

---
## 2. Theorems you can cite

| Topic | Fact (`ha.…`) | Statement |
|---|---|---|
| Coefficients | `coeff_const_mul`, `coeff_add` | linearity |
| | `coeff_derivative` | $(f')^\wedge(n) = in\,\hat f(n)$ for $f(-\pi)=f(\pi)$ (integration by parts) |
| $L^2$ theory | `bessel_summable` | $\sum_n|\hat f(n)|^2<\infty$ for $f\in L^2$ (Bessel) |
| | `parseval` | $\sum_n|\hat f(n)|^2 = \frac1{2\pi}\int_{-\pi}^{\pi}|f|^2$ |
| | `riemann_lebesgue_coeff` | $\hat f(n)\to0$ as $|n|\to\infty$ for $f\in L^2$ |
| Transform | `riemann_lebesgue` | $\hat f(\xi)\to0$ as $|\xi|\to\infty$ |
| | `gaussian` | $\widehat{e^{-\pi x^2}} = e^{-\pi\xi^2}$ |
| | `inversion` | $f(x) = (\hat f)^\vee(x)$ for continuous $f$ with $f,\hat f\in L^1$ |
| | `ift_at_zero` | $F^\vee(0) = \int F$ |
| Small facts | `terms_tend_to_zero`, `tendsto_zero_of_norm_sq`, `norm_I_mul_int`, `abs_int_ne_zero`, `div_of_mul`, `div_div`, `sum_of_zero_coeffs` | for the routine steps |

---
## 3. Pictures

**Partial sums of a Fourier series.** For the square wave, $S_N f(x) = \frac4\pi\sum_{k\ \mathrm{odd}\le N}
\frac{\sin kx}{k}$. Increase N: the partial sums converge at every point of continuity, but the overshoot
near the jump does not go away (the Gibbs phenomenon).

In [ ]:
show(*figure_partial_sums(9))

In [ ]:
explore_partial_sums()

**Smoothness ⇔ decay.** The coefficients of a function with a jump decay like $1/n$, with a corner like
$1/n^2$, and of a smooth function faster than any power — Example 1 below proves the first step of this.

In [ ]:
show(figure_decay())

**Riemann–Lebesgue.** Multiplying by a fast oscillation makes the integral cancel out.

In [ ]:
show(*figure_riemann_lebesgue(8))

In [ ]:
explore_riemann_lebesgue()

**The Gaussian.** $e^{-\pi a x^2}$ has Fourier transform $a^{-1/2}e^{-\pi\xi^2/a}$; for $a=1$ it is its own
transform (`ha.gaussian`). Squeezing $f$ spreads $\hat f$.

In [ ]:
show(*figure_gaussian(1.0))

In [ ]:
explore_gaussian()

---
## 4. Worked examples

### Example 1. Smoothness gives decay

**Theorem.** If $f' = g$ on $\mathbb R$, $g\in L^1(-\pi,\pi)$, $f(-\pi) = f(\pi)$ and $n\neq0$, then
$|\hat f(n)| = |\hat g(n)|/|n|$.

**In a book.** Integrating by parts, the boundary terms cancel because $f(-\pi)=f(\pi)$, so
$\hat g(n) = in\,\hat f(n)$. Taking absolute values, $|\hat g(n)| = |n|\,|\hat f(n)|$, and since $n\ne0$,
$|\hat f(n)| = |\hat g(n)|/|n|$. ∎

In [ ]:
example_1 = decay(
    pl.claim(ha.coeff(g, n) == ca.I() * Cx(n) * ha.coeff(f, n), because=ha.coeff_derivative,
             note="integration by parts; the boundary terms cancel since f(−π) = f(π)"),
    pl.chain(pl.norm(ha.coeff(g, n)),
             ("=", pl.norm(ca.I() * Cx(n) * ha.coeff(f, n)), "step1"),
             ("=", pl.abs(R(n)) * pl.norm(ha.coeff(f, n)), ha.norm_I_mul_int),
             name="habs"),
    pl.claim(pl.abs(R(n)) != 0, because=[ha.abs_int_ne_zero, "hn"]),
    pl.conclude(because=ha.div_of_mul, note="divide by |n| ≠ 0"),
)
check(example_1)

### Example 2. The Riemann–Lebesgue lemma for $L^2$ functions

**In a book.** By Bessel's inequality $\sum_n|\hat f(n)|^2$ converges, so its terms tend to $0$; hence
$\hat f(n)\to0$ as $|n|\to\infty$. ∎

In [ ]:
example_2 = riemann_lebesgue_l2(
    pl.claim(ha.summable(pl.fun(lambda n: pl.norm(ha.coeff(f, n)) ** 2, pl.Int)),
             because=ha.bessel_summable),
    pl.claim(ha.tends_to_zero(pl.fun(lambda n: pl.norm(ha.coeff(f, n)) ** 2, pl.Int)),
             because=ha.terms_tend_to_zero),
    pl.conclude(because=ha.tendsto_zero_of_norm_sq),
)
check(example_2)

### Example 3. A function is determined by its Fourier transform

**Theorem.** If $f, g$ are continuous, $f, g, \hat f, \hat g \in L^1(\mathbb R)$ and $\hat f = \hat g$,
then $f = g$.

**In a book.** By Fourier inversion, $f(x) = (\hat f)^\vee(x) = (\hat g)^\vee(x) = g(x)$. ∎

In [ ]:
example_3 = uniqueness(
    pl.fix(x),
    pl.claim(g(x) == ha.ift(ha.ft_fun(g), x), because=ha.inversion, name="hgx"),
    pl.chain(f(x),
             ("=", ha.ift(ha.ft_fun(f), x), ha.inversion, "Fourier inversion for f"),
             ("=", ha.ift(ha.ft_fun(g), x), "heq", "f̂ = ĝ"),
             ("=", g(x), "hgx", "Fourier inversion for g")),
)
check(example_3)

---
## 5. A wrong step

Forgetting the hypothesis $n \neq 0$ when dividing: here we cite the fact for $|n|\ne0$ without the
hypothesis `hn`, in a statement where $n$ may be $0$. Lean refuses step 3.

In [ ]:
from harmonic_lecture import f as F_, g as G_
bad = pl.Theorem(
    "decay_without_n_ne_zero", [F_, G_, n],
    assumptions={"hd": ha.derivative(F_, G_), "hg": ha.integrable_on_circle(G_),
                 "hper": F_(-pl.analysis.pi()) == F_(pl.analysis.pi())},
    conclusion=pl.norm(ha.coeff(F_, n)) == pl.norm(ha.coeff(G_, n)) / pl.abs(R(n)),
    proof=pl.proof(
        pl.claim(ha.coeff(G_, n) == ca.I() * Cx(n) * ha.coeff(F_, n), because=ha.coeff_derivative),
        pl.chain(pl.norm(ha.coeff(G_, n)),
                 ("=", pl.norm(ca.I() * Cx(n) * ha.coeff(F_, n)), "step1"),
                 ("=", pl.abs(R(n)) * pl.norm(ha.coeff(F_, n)), ha.norm_I_mul_int)),
        pl.claim(pl.abs(R(n)) != 0, because=ha.abs_int_ne_zero),      # ✗ needs n ≠ 0
        pl.conclude(because=ha.div_of_mul),
    ),
)
check(bad)

(Integration by parts, step 1, also needs $n \ne 0$ — Lean flags every step that uses it.)

---
## 6. Exercises

### Exercise 1. Two derivatives give $1/n^2$ decay

If $f' = g$, $g' = h$, $g, h \in L^1(-\pi,\pi)$, $f(\pm\pi)$ and $g(\pm\pi)$ agree, and $n\neq0$, show
$|\hat f(n)| = |\hat h(n)|/(|n|\,|n|)$.

<details><summary><b>Hint</b></summary>

Cite **Example 1** twice: `because=example_1` proves $|\hat f(n)| = |\hat g(n)|/|n|$ and
$|\hat g(n)| = |\hat h(n)|/|n|$ (Lean instantiates it). Then chain, and finish with `ha.div_div`.
</details>

In [ ]:
my_proof_1 = exercise_1(
    # pl.claim(pl.norm(ha.coeff(f, n)) == pl.norm(ha.coeff(g, n)) / pl.abs(R(n)), because=example_1),
    # ...
)
check(my_proof_1)

### Exercise 2. Vanishing coefficients

If $f\in L^2(-\pi,\pi)$ and $\hat f(n) = 0$ for every $n$, show $\frac1{2\pi}\int_{-\pi}^{\pi}|f|^2 = 0$.
(So $f = 0$ almost everywhere: the trigonometric system is complete.)

<details><summary><b>Hint</b></summary>

Parseval (`ha.parseval`) and `ha.sum_of_zero_coeffs`; Lean finishes the arithmetic.
</details>

In [ ]:
my_proof_2 = exercise_2(
    # ...
)
check(my_proof_2)

### Exercise 3. The value at the origin

If $f$ is continuous and $f, \hat f\in L^1(\mathbb R)$, show $f(0) = \int_{\mathbb R}\hat f(\xi)\,d\xi$.

<details><summary><b>Hint</b></summary>

One chain: $f(0) = (\hat f)^\vee(0)$ by `ha.inversion`, $= \int \hat f$ by `ha.ift_at_zero`.
</details>

In [ ]:
my_proof_3 = exercise_3(
    # pl.chain(f(0), ...),
)
check(my_proof_3)

---
## 7. What is not covered yet

Mathlib (at the version PyLeanX pins) has the $L^2$ theory of Fourier series, the Fourier transform with
Riemann–Lebesgue, inversion, Plancherel on $L^2$, Poisson summation and the Gaussian, convolution, and
Lebesgue differentiation. It does **not** yet have the Fejér and Dirichlet kernels and summability
methods, the Hardy–Littlewood maximal inequality, interpolation (Riesz–Thorin, Marcinkiewicz), the
Hilbert transform or Calderón–Zygmund theory — the core of Stein's *Harmonic Analysis* and of Tao's later
notes. Proofs that need them cannot be checked until those results are formalised; PyLeanX will not
pretend otherwise.

To write your own theorem: `pl.Theorem(name, variables, assumptions={...}, conclusion=..., proof=pl.proof(...))`
with the vocabulary above, then `check(...)`.

---
## 8. Solutions (each checked by Lean)

`show_solution(1)` … `show_solution(3)` re-checks one.

<details><summary><b>Solution 1</b></summary>

```python
exercise_1(
    pl.claim(pl.norm(ha.coeff(f, n)) == pl.norm(ha.coeff(g, n)) / pl.abs(R(n)),
             because=example_1, note="Example 1 for f"),
    pl.claim(pl.norm(ha.coeff(g, n)) == pl.norm(ha.coeff(h, n)) / pl.abs(R(n)),
             because=example_1, note="Example 1 for f′"),
    pl.chain(pl.norm(ha.coeff(f, n)),
             ("=", pl.norm(ha.coeff(g, n)) / pl.abs(R(n)), "step1"),
             ("=", pl.norm(ha.coeff(h, n)) / pl.abs(R(n)) / pl.abs(R(n)), "step2"),
             ("=", pl.norm(ha.coeff(h, n)) / (pl.abs(R(n)) * pl.abs(R(n))), ha.div_div)),
)
```
</details>

<details><summary><b>Solution 2</b></summary>

```python
exercise_2(
    pl.claim(ha.coeff_sq_sum(f) == ha.mean_square(f), because=ha.parseval),
    pl.claim(ha.coeff_sq_sum(f) == 0, because=ha.sum_of_zero_coeffs),
    pl.conclude(),
)
```
</details>

<details><summary><b>Solution 3</b></summary>

```python
exercise_3(
    pl.chain(f(0),
             ("=", ha.ift(ha.ft_fun(f), 0), ha.inversion, "Fourier inversion"),
             ("=", pl.integral(lambda ξ: ha.ft(f, ξ)), ha.ift_at_zero)),
)
```
</details>